In [2]:
import pandas as pd    # For data manipulation and analysis
df = pd.read_csv("student_habits_performance.csv")  # Reading the dataset from CSV file

In [ ]:
df.head()  # Displaying the first 5 rows of the dataset

In [4]:
import numpy as np  # Importing numpy for numerical operations
import matplotlib.pyplot as plt  # Importing matplotlib for plotting (note: common alias is plt)
import seaborn as sns  # Importing seaborn for advanced visualization


In [5]:
sns.set(style="whitegrid")   # Setting seaborn style for better plot aesthetics

In [ ]:
df.isna().sum()   # Checking for missing values in each column

In [ ]:
df.duplicated().sum()  # Checking for duplicate rows in the dataset


In [ ]:
df.describe()  # Getting statistical summary of numerical columns


In [ ]:
df.describe(include="object").columns  # Getting names of categorical columns (object dtype)


In [10]:
# Defining list of categorical columns for further analysis

catagorical_cols=['student_id', 'gender', 'part_time_job', 'diet_quality',
       'parental_education_level', 'internet_quality',
       'extracurricular_participation']


In [ ]:
# Looping through each categorical column to show value counts

for col in catagorical_cols:
    print(f"Value counts for {col}: \n {df[col].value_counts()}")   # Displaying frequency of each category

In [ ]:
# Histogram of all numerical features

df.hist(bins=20,edgecolor="black")
plt.tight_layout
plt.show()

In [ ]:
# Count plots for categorical variables

for col in catagorical_cols:
    sns.countplot(data=df,x=col)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
# Correlation heatmap for numerical features

sns.heatmap(df.corr(numeric_only=True),annot=True,cmap="coolwarm",fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

In [ ]:

df.describe().columns

In [16]:
# Define list of numerical features for scatter plotting

num_features=['age', 'study_hours_per_day', 'social_media_hours', 'netflix_hours',
       'attendance_percentage', 'sleep_hours', 'exercise_frequency',
       'mental_health_rating']

In [ ]:
# Scatter plot of numerical features vs target

for feature in num_features:
    sns.scatterplot(data=df , x = feature ,y = "exam_score")
    plt.title(f"{feature} VS Exam Score")
    plt.show()

In [ ]:
# Box plots to analyze exam score distribution across categories

for col in catagorical_cols:
    sns.boxplot(data=df , x = col ,y = "exam_score")
    plt.title(f"Exam Score By {col}")
    plt.show()

In [19]:
from sklearn.model_selection import train_test_split, GridSearchCV
# train_test_split: to split data into training and test sets
# GridSearchCV: to perform hyperparameter tuning using cross-validation

from sklearn.preprocessing import LabelEncoder
# LabelEncoder: to convert categorical labels (e.g., 'Yes'/'No') into numeric form (e.g., 1/0)

from sklearn.metrics import mean_squared_error, r2_score
# mean_squared_error: to calculate the RMSE (error metric)
# r2_score: to measure how well the model explains the variance (model performance)

from sklearn.linear_model import LinearRegression, Ridge
# LinearRegression: simple linear model
# Ridge: regularized version of linear regression to avoid overfitting

from sklearn.ensemble import RandomForestRegressor
# RandomForestRegressor: ensemble model using multiple decision trees for better performance

from sklearn.tree import DecisionTreeRegressor
# DecisionTreeRegressor: tree-based model that splits data based on feature thresholds


In [ ]:
df.columns

In [21]:
# Selecting features and target variable

feature = ["study_hours_per_day","attendance_percentage","mental_health_rating","sleep_hours","part_time_job"]
target ="exam_score"


In [23]:
# Prepare model-ready DataFrame

df_model=df[feature+[target]].copy()

In [ ]:
df_model

In [25]:
# Encode categorical variable "part_time_job" (Yes/No -> 1/0)

le = LabelEncoder()
df_model["part_time_job"]=le.fit_transform(df_model["part_time_job"])

In [ ]:
df_model

In [28]:
# Split data into features and target

X = df_model[feature]
Y = df_model[target]

In [30]:
# Train/test split

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)


In [ ]:
# Confirm size of splits

len(y_test)

In [ ]:
len(y_train)

In [33]:
# Define dictionary of models and their hyperparameters for GridSearchCV

models = {
    "LinearRegression":{
        "model": LinearRegression(),
        "params": {}
                        },
    "Decisiontree": {
        "model": DecisionTreeRegressor(),
        "params": {"max_depth" : [3,5,10],"min_samples_split":[2,5]}
                    },      

    "RandomForest": {
        "model" : RandomForestRegressor(),
        "params":{"n_estimators": [50,100], "max_depth" :[5,10]}
                    }
        }

In [34]:
# List to store best model results

best_models =[]



In [ ]:
# Loop through each model and perform GridSearchCV

for name,config in models.items():
    print(f"Training {name}")
    grid = GridSearchCV(config["model"], config["params"], cv=5, scoring="neg_mean_squared_error")
    grid.fit(x_train,y_train)

    y_pred = grid.predict(x_test)
    rmse = np.sqrt(mean_squared_error(y_test,y_pred)) # Root Mean Squared Error
    r2 = r2_score(y_test,y_pred) # R² Score
    best_models.append({
        "model": name,
        "best_params" : grid.best_params_,
        "rmse" : rmse,
        "r2":r2
    })



In [ ]:
best_models

In [ ]:
# Convert results to DataFrame for comparison

results_df = pd.DataFrame(best_models)
results_df

In [ ]:
# Sort models by RMSE to find the best

results_df.sort_values(by="rmse")

In [40]:
import joblib # For saving/loading models

# Select the best model based on lowest RMSE
best_row = results_df.sort_values(by="rmse").iloc[0]


In [ ]:
best_row

In [45]:
best_model_name = best_row["model"]


In [ ]:
best_model_name

In [48]:
# Retrieve the model configuration

best_model_config = models[best_model_name]

In [ ]:
best_model_config

In [52]:
# Train the final model on the full dataset

final_model = best_model_config["model"]

In [ ]:
final_model.fit(X,Y)

In [ ]:
# Save the final model to a .pkl file

joblib.dump(final_model,"best_model.pkl")

In [ ]:
# Example: Load the saved model and make predictions

joblib.load("best_model.pkl").predict(x_test)